# 05 CNN + VLM pseudo-labels (GPU)
Needs notebook 03 (pseudo-label files). 5 folds x {5, 10, 25}% labels.

In [ ]:
import glob, os, sys
if os.path.isdir("/kaggle/input"):
    hits = glob.glob("/kaggle/input/**/src/woundvlm/__init__.py", recursive=True)
    assert hits, "attach the project dataset (lowerlimbwoundvlm) as an input"
    SRC = hits[0].rsplit("/woundvlm/", 1)[0]
else:
    SRC = os.path.abspath("../src")
sys.path.insert(0, SRC)
from woundvlm import env
E = env.setup()
P = E["project"]
print(E)

In [ ]:
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
assert DEVICE == "cuda" or not E["in_kaggle"], "select a GPU accelerator (T4) for this notebook" 

In [ ]:
assert glob.glob(os.path.join(P, 'outputs', 'heads', 'pseudo', '*.csv')), 'attach notebook 03 output'
from woundvlm import experiments
experiments.run_cnn(P, 'pseudo', device=DEVICE)

In [ ]:
import json, pandas as pd
rows = [json.load(open(f)) for f in sorted(glob.glob(os.path.join(P, 'outputs', 'cnn', 'cnnpl_*.json')))]
t = pd.DataFrame([{k: r[k] for k in ('tag', 'n_fit', 'n_pseudo', 'seconds')} for r in rows])
t.to_csv(os.path.join(E['results'], '05_cnn_pseudo_runs.csv'), index=False); t